In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from pathlib import Path

In [2]:
import time
from pathlib import Path

import soundfile as sf
import torch
from tqdm import tqdm
from torch import multiprocessing

import sys

In [3]:
stored_files = list(Path('../../Documents/mila-human-wav-txt').glob('*.WAV'))
stored_files

[PosixPath('../../Documents/mila-human-wav-txt/20220826_070000.WAV'),
 PosixPath('../../Documents/mila-human-wav-txt/20220727_083000.WAV'),
 PosixPath('../../Documents/mila-human-wav-txt/20220829_090000.WAV'),
 PosixPath('../../Documents/mila-human-wav-txt/20220730_053000.WAV')]

In [4]:
import batdetect2.api as api

Used `python3 -m pip install batdetect2`

`Successfully installed batdetect2-1.1.1 click-8.1.8 filelock-3.16.1 librosa-0.10.2.post1 matplotlib-3.9.4 mpmath-1.3.0 numpy-1.24.4 nvidia-cublas-cu12-12.1.3.1 nvidia-cuda-cupti-cu12-12.1.105 nvidia-cuda-nvrtc-cu12-12.1.105 nvidia-cuda-runtime-cu12-12.1.105 nvidia-cudnn-cu12-9.1.0.70 nvidia-cufft-cu12-11.0.2.54 nvidia-curand-cu12-10.3.2.106 nvidia-cusolver-cu12-11.4.5.107 nvidia-cusparse-cu12-12.1.0.106 nvidia-nccl-cu12-2.20.5 nvidia-nvjitlink-cu12-12.6.85 nvidia-nvtx-cu12-12.1.105 pandas-2.2.3 scikit-learn-1.6.0 scipy-1.13.1 soxr-0.5.0.post1 sympy-1.13.3 torch-2.4.1 torchaudio-2.4.1 torchvision-0.19.1 triton-3.0.0 typing-extensions-4.12.2`

In [5]:
from batdetect2.detector.parameters import (
    DEFAULT_MODEL_PATH,
    DEFAULT_PROCESSING_CONFIGURATIONS,
    DEFAULT_SPECTROGRAM_PARAMETERS,
    TARGET_SAMPLERATE_HZ,
)

In [6]:
TARGET_SAMPLERATE_HZ

256000

In [7]:
DEFAULT_SPECTROGRAM_PARAMETERS

{'fft_win_length': 0.002,
 'fft_overlap': 0.75,
 'spec_height': 256,
 'resize_factor': 0.5,
 'spec_divide_factor': 32,
 'max_freq': 120000,
 'min_freq': 10000,
 'spec_scale': 'pcen',
 'denoise_spec_avg': True,
 'max_scale_spec': False}

In [8]:
DEFAULT_PROCESSING_CONFIGURATIONS

{'detection_threshold': 0.01,
 'spec_slices': False,
 'chunk_size': 3,
 'spec_features': False,
 'cnn_features': False,
 'quiet': True,
 'target_samp_rate': 256000,
 'fft_win_length': 0.002,
 'fft_overlap': 0.75,
 'resize_factor': 0.5,
 'spec_divide_factor': 32,
 'spec_height': 256,
 'scale_raw_audio': False,
 'class_names': [],
 'time_expansion': 1,
 'top_n': 3,
 'return_raw_preds': False,
 'max_duration': None,
 'nms_kernel_size': 9,
 'max_freq': 120000,
 'min_freq': 10000,
 'nms_top_k_per_sec': 200,
 'spec_scale': 'pcen',
 'denoise_spec_avg': True,
 'max_scale_spec': False}

In [9]:
# Use GPU if available
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Default model
MODEL, PARAMS = api.load_model(DEFAULT_MODEL_PATH, device=DEVICE)

In [10]:
PARAMS

{'data_dir': '/data1/bat_data/data/',
 'ann_dir': '/data1/bat_data/annotations/anns_same/',
 'train_split': 'same',
 'standardize_classs_names_ip': 'Rhinolophus ferrumequinum;Rhinolophus hipposideros',
 'model_name': 'Net2DFast',
 'num_filters': 128,
 'experiment': '../../experiments/2021_12_13__20_20_37/',
 'model_file_name': '../../experiments/2021_12_13__20_20_37/2021_12_13__20_20_37.pth.tar',
 'op_im_dir': '../../experiments/2021_12_13__20_20_37/op_ims/',
 'op_im_dir_test': '../../experiments/2021_12_13__20_20_37/op_ims_test/',
 'notes': '',
 'target_samp_rate': 256000,
 'fft_win_length': 0.002,
 'fft_overlap': 0.75,
 'max_freq': 120000,
 'min_freq': 10000,
 'resize_factor': 0.5,
 'spec_height': 256,
 'spec_train_width': 512,
 'spec_divide_factor': 32,
 'denoise_spec_avg': True,
 'scale_raw_audio': False,
 'max_scale_spec': False,
 'spec_scale': 'pcen',
 'detection_overlap': 0.01,
 'ignore_start_end': 0.01,
 'detection_threshold': 0.01,
 'nms_kernel_size': 9,
 'nms_top_k_per_sec': 

In [ ]:
# Default processing configuration
ubna_config = api.get_config(detection_threshold=self.detection_threshold,
                                spec_slices = self.spec_slices,
                                chunk_size = self.chunk_size,
                                time_expansion_factor = self.time_expansion_factor
                                quiet = self.quiet,
                                cnn_features = self.cnn_features)

In [12]:
# Default processing configuration
ubna_config = api.get_config(detection_threshold=0.5)

In [16]:
'alpha' in ubna_config.keys()

False

In [12]:
pd.DataFrame([ubna_config]).iloc[:,:10]

,detection_threshold,spec_slices,chunk_size,spec_features,cnn_features,quiet,target_samp_rate,fft_win_length,fft_overlap,resize_factor
0,0.5,False,3,False,False,True,256000,0.002,0.75,0.5


In [13]:
pd.DataFrame([ubna_config]).iloc[:,10:20]

,spec_divide_factor,spec_height,scale_raw_audio,class_names,time_expansion,top_n,return_raw_preds,max_duration,nms_kernel_size,max_freq
0,32,256,False,"[Barbastellus barbastellus, Eptesicus serotinu...",1,3,False,None,9,120000


In [14]:
pd.DataFrame([ubna_config]).iloc[:,20:30]

,min_freq,nms_top_k_per_sec,spec_scale,denoise_spec_avg,max_scale_spec,data_dir,ann_dir,train_split,standardize_classs_names_ip,model_name
0,10000,200,pcen,True,False,/data1/bat_data/data/,/data1/bat_data/annotations/anns_same/,same,Rhinolophus ferrumequinum;Rhinolophus hipposid...,Net2DFast


In [15]:
pd.DataFrame([ubna_config]).iloc[:,30:40]

,num_filters,experiment,model_file_name,op_im_dir,op_im_dir_test,notes,spec_train_width,detection_overlap,ignore_start_end,target_sigma
0,128,../../experiments/2021_12_13__20_20_37/,../../experiments/2021_12_13__20_20_37/2021_12...,../../experiments/2021_12_13__20_20_37/op_ims/,../../experiments/2021_12_13__20_20_37/op_ims_...,,512,0.01,0.01,2.0


In [16]:
pd.DataFrame([ubna_config]).iloc[:,40:50]

,aug_prob,augment_at_train,augment_at_train_combine,echo_max_delay,stretch_squeeze_delta,mask_max_time_perc,mask_max_freq_perc,spec_amp_scaling,aug_sampling_rates,train_loss
0,0.2,True,True,0.005,0.04,0.05,0.1,2.0,"[220500, 256000, 300000, 312500, 384000, 44100...",focal


In [17]:
pd.DataFrame([ubna_config]).iloc[:,50:60]

,det_loss_weight,size_loss_weight,class_loss_weight,individual_loss_weight,emb_dim,lr,batch_size,num_workers,num_epochs,num_eval_epochs
0,1.0,0.1,2.0,0.0,0,0.001,8,4,200,5


In [18]:
# Process audio file
results = api.process_file(stored_files[0], config=ubna_config)

KeyboardInterrupt: 

In [75]:
results['pred_dict']['annotation']

[{'start_time': 0.0015,
  'end_time': 0.0328,
  'low_freq': 68437,
  'high_freq': 79512,
  'class': 'Rhinolophus ferrumequinum',
  'class_prob': 0.028,
  'det_prob': 0.028,
  'individual': '-1',
  'event': 'Echolocation'},
 {'start_time': 0.0015,
  'end_time': 0.0098,
  'low_freq': 47812,
  'high_freq': 51963,
  'class': 'Pipistrellus pipistrellus',
  'class_prob': 0.029,
  'det_prob': 0.035,
  'individual': '-1',
  'event': 'Echolocation'},
 {'start_time': 0.0015,
  'end_time': 0.0096,
  'low_freq': 22890,
  'high_freq': 29195,
  'class': 'Nyctalus leisleri',
  'class_prob': 0.011,
  'det_prob': 0.033,
  'individual': '-1',
  'event': 'Echolocation'},
 {'start_time': 0.0685,
  'end_time': 0.0826,
  'low_freq': 22890,
  'high_freq': 27036,
  'class': 'Nyctalus leisleri',
  'class_prob': 0.014,
  'det_prob': 0.025,
  'individual': '-1',
  'event': 'Echolocation'},
 {'start_time': 0.0725,
  'end_time': 0.0786,
  'low_freq': 11718,
  'high_freq': 64612,
  'class': 'Myotis nattereri',
  'c

In [37]:
pd.DataFrame(results['pred_dict']['annotation'])

,start_time,end_time,low_freq,high_freq,class,class_prob,det_prob,individual,event
0,0.0015,0.0328,68437,79512,Rhinolophus ferrumequinum,0.028,0.028,-1,Echolocation
1,0.0015,0.0098,47812,51963,Pipistrellus pipistrellus,0.029,0.035,-1,Echolocation
2,0.0015,0.0096,22890,29195,Nyctalus leisleri,0.011,0.033,-1,Echolocation
3,0.0685,0.0826,22890,27036,Nyctalus leisleri,0.014,0.025,-1,Echolocation
4,0.0725,0.0786,11718,64612,Myotis nattereri,0.010,0.025,-1,Echolocation
...,...,...,...,...,...,...,...,...,...
92869,1794.9456,1794.9827,68437,81974,Rhinolophus ferrumequinum,0.023,0.023,-1,Echolocation
92870,1794.9504,1794.9609,22031,27451,Nyctalus leisleri,0.011,0.031,-1,Echolocation
92871,1794.9855,1794.9958,22890,29719,Nyctalus leisleri,0.009,0.025,-1,Echolocation
92872,1795.0225,1795.0581,68437,80499,Rhinolophus ferrumequinum,0.029,0.029,-1,Echolocation
